In [1]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
import time

In [2]:
prod_url = pd.read_csv('prod_urls.csv')

In [3]:
prod_url = prod_url.drop(columns=['status', 'discovered_at'])
prod_url['product_id'] = prod_url['product_id'].astype(int)
prod_url.head(5)

,product_id,url
0,698859,https://www.naiin.com/product/detail/698859/
1,698857,https://www.naiin.com/product/detail/698857/
2,698856,https://www.naiin.com/product/detail/698856/
3,699421,https://www.naiin.com/product/detail/699421/
4,698858,https://www.naiin.com/product/detail/698858/


In [4]:
examples = prod_url.head(5)
examples

,product_id,url
0,698859,https://www.naiin.com/product/detail/698859/
1,698857,https://www.naiin.com/product/detail/698857/
2,698856,https://www.naiin.com/product/detail/698856/
3,699421,https://www.naiin.com/product/detail/699421/
4,698858,https://www.naiin.com/product/detail/698858/


In [ ]:
import re
import json
import html

def extract_extra_info(html_content):
    extra_data = {}
    # ใช้ BeautifulSoup ช่วยสำหรับส่วนที่ JSON ไม่มีข้อมูล
    soup = BeautifulSoup(html_content, 'lxml')
    
    try:
        # 1. ดึง Rating และข้อมูลเบื้องต้นจาก JSON (วิธีเดิมที่คุณใช้แล้วได้ผล)
        match = re.search(r'AverageRating&quot;:(.+?)\}', html_content)
        if match:
            raw_json = '{"AverageRating":' + match.group(1) + '}'
            clean_json = html.unescape(raw_json)
            json_obj = json.loads(clean_json)
            
            extra_data = {
                'AverageRating': json_obj.get('AverageRating') or None,
                'TotalRating': json_obj.get('TotalRating') or None,
                'NumberOfPage': json_obj.get('NumberOfPage') or None,
                # เริ่มต้นด้วย None เดี๋ยวเราใช้ BeautifulSoup ทับถ้าหาเจอ
                'Width': None,
                'Height': None,
                'Thick': None,
                'GrossWeightKG': None,
                'FileSizeMB': None
            }

        # 2. ใช้ BeautifulSoup เจาะหา "ขนาดไฟล์", "ขนาด", "น้ำหนัก" จาก Label โดยตรง
        # เราจะหา <label> ที่มีข้อความที่ต้องการ แล้วหา <p> ที่อยู่ถัดจากมัน
        labels = soup.find_all('label', class_='product-label')
        for label in labels:
            text = label.get_text(strip=True)
            detail_p = label.find_next_sibling('p', class_='product-label-detail')
            
            if detail_p:
                val = detail_p.get_text(strip=True)
                
                if "ขนาดไฟล์" in text:
                    # ดึงเฉพาะตัวเลขจาก "4.08 MB"
                    size_val = re.search(r'[\d\.]+', val)
                    if size_val: extra_data['FileSizeMB'] = size_val.group()
                
                elif "ขนาด" in text:
                    # แยก "0 x 0 x 0 CM" ออกเป็น 3 ส่วน
                    dims = re.findall(r'[\d\.]+', val)
                    if len(dims) >= 3:
                        extra_data['Width'] = dims[0]
                        extra_data['Height'] = dims[1]
                        extra_data['Thick'] = dims[2]
                
                elif "น้ำหนัก" in text:
                    # ดึงเฉพาะตัวเลขจาก "0.374 KG"
                    weight_val = re.search(r'[\d\.]+', val)
                    if weight_val: extra_data['GrossWeightKG'] = weight_val.group()

    except Exception as e:
        print(f"  [Warning] Extra info skip: {e}")
        
    return extra_data

In [14]:
def scrape(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Referer': 'https://www.naiin.com/',
    }
    
    time.sleep(1.5) # เพิ่มเวลาหน่อยกันโดนบล็อก
    response = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(response.text, 'lxml')
    
    # ข้อมูลพื้นฐานจาก Metadata
    data = {'url': url}
    meta_map = {
        'Title': soup.find('meta', property='og:title'),
        'Price_Full': soup.find('meta', property='og:product:price:amount'),
        'Price_Sale': soup.find('meta', property='og:product:sale_price:amount'),
        'Barcode': soup.find('meta', property='book:isbn'),
        'Release_Date': soup.find('meta', property='book:release_date'),
        'keywords': soup.find('meta', attrs={'name': 'keywords'}),
    }

    for key, tag in meta_map.items():
        data[key] = tag['content'] if tag else "N/A"

    # --- ส่วนที่แก้ปัญหา Error ---
    # ส่ง response.text (String) เข้าไป
    extra_info = extract_extra_info(response.text)
    
    # รวมข้อมูลเข้าด้วยกัน
    data.update(extra_info)
    
    return data

In [15]:
import time

# 1. บันทึกเวลาเริ่มต้น
start_time = time.time()
print(f"🚀 Started at: {time.strftime('%H:%M:%S', time.localtime(start_time))}")

test_results = []

# ทดสอบกับตัวอย่าง (สมมติเอา 5 แถวแรก)
for _, row in examples.head(5).iterrows():
    print(f"Scraping ID: {row['product_id']} | URL: {row['url']}")
    
    # รันฟังก์ชัน Scrape
    res = scrape(row['url'])
    
    # แทรก product_id ไว้ที่ตำแหน่งแรก
    final_row = {'product_id': row['product_id']}
    final_row.update(res)
    
    test_results.append(final_row)

# 2. บันทึกเวลาสิ้นสุด
end_time = time.time()
duration = end_time - start_time # วินาที

print("-" * 30)
print(f"🏁 Finished at: {time.strftime('%H:%M:%S', time.localtime(end_time))}")
print(f"⏱️ Total Duration: {duration:.2f} seconds")

# 3. คำนวณค่าเฉลี่ย (ช่วยในการประเมินเวลาสำหรับงานแสนรายการ)
avg_time = duration / len(test_results)
print(f"📊 Average time per item: {avg_time:.2f} seconds")

🚀 Started at: 21:18:46
Scraping ID: 698859 | URL: https://www.naiin.com/product/detail/698859/
Scraping ID: 698857 | URL: https://www.naiin.com/product/detail/698857/
Scraping ID: 698856 | URL: https://www.naiin.com/product/detail/698856/
Scraping ID: 699421 | URL: https://www.naiin.com/product/detail/699421/
Scraping ID: 698858 | URL: https://www.naiin.com/product/detail/698858/
------------------------------
🏁 Finished at: 21:18:56
⏱️ Total Duration: 10.47 seconds
📊 Average time per item: 2.09 seconds


In [16]:
test_results

[{'product_id': 698859,
  'url': 'https://www.naiin.com/product/detail/698859/',
  'Title': 'สัญญาณเตือนตาย เล่ม 3E-Books | e-book ร้านหนังสือนายอินทร์',
  'Price_Full': '335',
  'Price_Sale': '285',
  'Barcode': '9786161888145',
  'Release_Date': '2026-03-11',
  'keywords': 'สัญญาณเตือนตาย เล่ม 3, โจวเฮ่าฮุย, prism publishing , ปริซึม, นิยายแปล, นิยายแปล, ',
  'AverageRating': None,
  'TotalRating': None,
  'NumberOfPage': 417,
  'Width': None,
  'Height': None,
  'Thick': None,
  'GrossWeightKG': None,
  'FileSizeMB': '4.66'},
 {'product_id': 698857,
  'url': 'https://www.naiin.com/product/detail/698857/',
  'Title': 'สัญญาณเตือนตาย เล่ม 1E-Books | e-book ร้านหนังสือนายอินทร์',
  'Price_Full': '315',
  'Price_Sale': '268',
  'Barcode': '9786161888176',
  'Release_Date': '2026-03-11',
  'keywords': 'สัญญาณเตือนตาย เล่ม 1, โจวเฮ่าฮุย, prism publishing , ปริซึม, นิยายแปล, นิยายแปล, ',
  'AverageRating': None,
  'TotalRating': None,
  'NumberOfPage': 377,
  'Width': None,
  'Height': Non

In [17]:
pd.DataFrame(test_results)

,product_id,url,Title,Price_Full,Price_Sale,Barcode,Release_Date,keywords,AverageRating,TotalRating,NumberOfPage,Width,Height,Thick,GrossWeightKG,FileSizeMB
0,698859,https://www.naiin.com/product/detail/698859/,สัญญาณเตือนตาย เล่ม 3E-Books | e-book ร้านหนัง...,335,285,9786161888145,2026-03-11,"สัญญาณเตือนตาย เล่ม 3, โจวเฮ่าฮุย, prism publi...",None,None,417.0,None,None,None,None,4.66
1,698857,https://www.naiin.com/product/detail/698857/,สัญญาณเตือนตาย เล่ม 1E-Books | e-book ร้านหนัง...,315,268,9786161888176,2026-03-11,"สัญญาณเตือนตาย เล่ม 1, โจวเฮ่าฮุย, prism publi...",None,None,377.0,None,None,None,None,4.08
2,698856,https://www.naiin.com/product/detail/698856/,สัญญาณเตือนตาย เล่ม 4E-Books | e-book ร้านหนัง...,345,293,9786161888152,2026-03-11,"สัญญาณเตือนตาย เล่ม 4, โจวเฮ่าฮุย, prism publi...",None,None,437.0,None,None,None,None,5.13
3,699421,https://www.naiin.com/product/detail/699421/,SET สัญญาณเตือนตาย เล่ม 1-5 (จบ)E-Books | e-bo...,1685,1432,9010129243,2026-03-13,"SET สัญญาณเตือนตาย เล่ม 1-5 (จบ), โจวเฮ่าฮุย, ...",None,None,NaN,None,None,None,None,0.00
4,698858,https://www.naiin.com/product/detail/698858/,สัญญาณเตือนตาย เล่ม 5 (เล่มจบ)E-Books | e-book...,345,293,9786161888169,2026-03-11,"สัญญาณเตือนตาย เล่ม 5 (เล่มจบ), โจวเฮ่าฮุย, pr...",None,None,417.0,None,None,None,None,4.70


In [20]:
scrape('https://www.naiin.com/product/detail/683989')

{'url': 'https://www.naiin.com/product/detail/683989',
 'Title': 'Self-Esteem 101 วัคซีนสำหรับใจในวันที่คุณคิดว่ายังไม่ดีพอBooks | ร้านหนังสือนายอินทร์',
 'Price_Full': '355',
 'Price_Sale': '302',
 'Barcode': '9786161882662',
 'Release_Date': '2025-09-25',
 'keywords': 'Self-Esteem 101 วัคซีนสำหรับใจในวันที่คุณคิดว่ายังไม่ดีพอ, ยุนฮงกยุน, อมรินทร์ How to, จิตวิทยา การพัฒนาตัวเอง, การพัฒนาตัวเอง how to, ',
 'AverageRating': 5,
 'TotalRating': 3,
 'NumberOfPage': 340,
 'Width': '0',
 'Height': '0',
 'Thick': '0',
 'GrossWeightKG': '0.386',
 'FileSizeMB': None}